# Compare the classifier paths

Train a direct feature classifier, then compare it with a frozen MiniHEP probe and a fine-tuned MiniHEP model. Each path uses the same fixed event partitions; only partition 0 trains the scaler and model. The model exports are JSON so they can travel with a submission. The encoder pretraining objective hides complete four-vectors and predicts them; labels are used only by the downstream classification head. MiniHEP is a small workshop prototype, not a general pretrained collider model.

The MiniHEP design is a fresh implementation inspired by the masked four-vector prototype in Gordon Watts's October 2026 HSF-India course repository, source commit `76cfa28c433d1f3e9ac95c0102974a55b37de54d`.

In [ ]:
from pathlib import Path
from higgslab.data import load_dataset
from higgslab.training import score, score_callback, train

dataset = Path('datasets/public')
events, pool, metadata = load_dataset(dataset)
print(f"Loaded {len(pool['label'])} events; partitions: {sorted(set(pool['partition']))}")

Use four epochs for the short CPU baseline. The three score arrays can be passed to the same template and fit code, so the comparison reaches the measurement rather than stopping at classifier accuracy.

In [ ]:
model_dir = Path('output/notebook-models')
model_paths = {
    mode: train(dataset, model_dir / f'{mode}.json', mode, epochs=4)
    for mode in ('supervised', 'minihep-frozen', 'minihep-finetuned')
}
scores = {mode: score(events, path) for mode, path in model_paths.items()}
{mode: (float(values.min()), float(values.max())) for mode, values in scores.items()}

The score callback lets the common fit use the selected model without changing its template construction or likelihood. The template partition stays outside training and can be used by the downstream analysis contract.

In [ ]:
callback = score_callback(model_paths['minihep-frozen'])
probe_scores = callback(events)
assert probe_scores.shape == (len(events['met_et']),)